In [1]:
#pip install requests beautifulsoup4 sentence-transformers chromadb 
#!pip install pymupdf

# Step 1 Output:
📋 Fetching sitemap: https://pvrea.coop/sitemap_index.xml
📋 Fetching sitemap: https://pvrea.coop/post-sitemap.xml
   Found 149 URLs
📋 Fetching sitemap: https://pvrea.coop/page-sitemap.xml
   Found 93 URLs
   Found 242 URLs

🔗 242 pages to scrape (capped at 250)

⏳ Waiting 2.6s...
✅ (1/242) Scraped: https://pvrea.coop/the-co-op/news/lineman-appreciation-people-power-our-community/

In [2]:
# scrape.py
import requests
import fitz  # pip install pymupdf
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import xml.etree.ElementTree as ET
import json, time, random, io

BASE_URL    = "https://pvrea.coop"
SITEMAP_URL = "https://pvrea.coop/sitemap_index.xml"

# ── Safety settings ────────────────────────────────────────────────
MAX_PAGES        = 300
MIN_DELAY        = 1.0
MAX_DELAY        = 3.0
REQUEST_TIMEOUT  = 10
MAX_PAGE_SIZE_KB = 500
MAX_PDF_SIZE_KB  = 5120   # 5 MB cap on PDFs
BOT_NAME         = "research-bot"
HEADERS = {
    "User-Agent": f"Mozilla/5.0 (compatible; {BOT_NAME}/1.0; +local-demo)"
}
# ───────────────────────────────────────────────────────────────────


# ── robots.txt ─────────────────────────────────────────────────────
def load_robots(base_url: str) -> RobotFileParser:
    """
    Fetch and parse robots.txt for the target domain.
    Falls back to a permissive parser if the file is unreachable,
    but logs a warning so you know enforcement may be incomplete.
    """
    rp = RobotFileParser()
    robots_url = urljoin(base_url, "/robots.txt")
    print(f"🤖 Fetching robots.txt: {robots_url}")
    try:
        r = requests.get(robots_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        if r.status_code == 200:
            rp.parse(r.text.splitlines())

            # ── Crawl-Delay support ────────────────────────────────
            # urllib.robotparser exposes crawl_delay() in Python 3.x
            delay = rp.crawl_delay(BOT_NAME) or rp.crawl_delay("*")
            if delay:
                global MIN_DELAY, MAX_DELAY
                # Honour the site's stated delay; keep our jitter on top
                MIN_DELAY = max(MIN_DELAY, float(delay))
                MAX_DELAY = max(MAX_DELAY, float(delay) + 2.0)
                print(f"   Crawl-Delay: {delay}s detected — adjusting delays")

            print(f"   robots.txt loaded successfully")
        else:
            print(f"⚠️  robots.txt returned HTTP {r.status_code} — proceeding permissively")
    except Exception as e:
        print(f"⚠️  Could not fetch robots.txt ({e}) — proceeding permissively")
    return rp


def is_allowed(rp: RobotFileParser, url: str) -> bool:
    """
    Return True only if robots.txt permits BOT_NAME (or *) to fetch url.
    Blocks the URL if the parser has no rules (fetch failed) for safety.
    """
    # can_fetch returns True when the parser has no rules, which would
    # silently allow everything — so we check both bot name and wildcard.
    return rp.can_fetch(BOT_NAME, url) and rp.can_fetch("*", url)
# ───────────────────────────────────────────────────────────────────


# ── PDF ingestion ──────────────────────────────────────────────────
def extract_pdf_from_url(pdf_url: str) -> str | None:
    """
    Download a PDF and extract its text with PyMuPDF.
    Returns plain text, or None if the PDF is unreachable / too large /
    appears to be scanned-only (no selectable text).
    """
    try:
        print(f"   📄 Fetching PDF: {pdf_url}")
        r = requests.get(
            pdf_url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT,
            stream=True
        )
        if r.status_code != 200:
            print(f"   ⚠️  PDF HTTP {r.status_code}: {pdf_url}")
            return None

        # Size guard — read up to MAX_PDF_SIZE_KB before bailing
        chunks = []
        total  = 0
        for chunk in r.iter_content(chunk_size=8192):
            total += len(chunk)
            if total > MAX_PDF_SIZE_KB * 1024:
                print(f"   ⏭️  PDF too large (>{MAX_PDF_SIZE_KB} KB): {pdf_url}")
                return None
            chunks.append(chunk)

        raw = b"".join(chunks)

        # Validate magic bytes — real PDFs start with %PDF
        if not raw.startswith(b"%PDF"):
            print(f"   ⏭️  Not a valid PDF (bad magic bytes): {pdf_url}")
            return None

        doc  = fitz.open(stream=raw, filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()

        text = text.strip()
        if len(text) < 100:
            # Likely a scanned / image-only PDF — skip rather than store noise
            print(f"   ⏭️  PDF appears to be image-only (no selectable text): {pdf_url}")
            return None

        print(f"   ✅ PDF extracted ({len(text):,} chars): {pdf_url}")
        return text

    except Exception as e:
        print(f"   ⚠️  PDF extraction error on {pdf_url}: {e}")
        return None


def find_pdf_links(soup: BeautifulSoup, page_url: str, rp: RobotFileParser) -> list[str]:
    """
    Return a deduplicated list of absolute PDF URLs found on a page,
    filtered to the same domain and allowed by robots.txt.
    """
    seen = set()
    pdf_links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href.lower().endswith(".pdf"):
            continue
        abs_url = urljoin(page_url, href)
        if urlparse(abs_url).netloc != urlparse(BASE_URL).netloc:
            continue  # off-domain PDF
        if abs_url in seen:
            continue
        if not is_allowed(rp, abs_url):
            print(f"   🚫 PDF blocked by robots.txt: {abs_url}")
            continue
        seen.add(abs_url)
        pdf_links.append(abs_url)
    return pdf_links
# ───────────────────────────────────────────────────────────────────


def get_urls_from_sitemap(sitemap_url):
    """
    Handles both sitemap indexes (which point to more sitemaps)
    and regular sitemaps (which point to pages).
    """
    urls = []
    try:
        print(f"📋 Fetching sitemap: {sitemap_url}")
        r = requests.get(sitemap_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        root = ET.fromstring(r.content)

        tag = root.tag.split("}")[-1] if "}" in root.tag else root.tag

        if tag == "sitemapindex":
            for sitemap in root.iter("{http://www.sitemaps.org/schemas/sitemap/0.9}loc"):
                child_urls = get_urls_from_sitemap(sitemap.text.strip())
                urls.extend(child_urls)
                time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        else:
            for loc in root.iter("{http://www.sitemaps.org/schemas/sitemap/0.9}loc"):
                urls.append(loc.text.strip())

        print(f"   Found {len(urls)} URLs")
    except Exception as e:
        print(f"⚠️  Sitemap error: {e}")
    return urls


def scrape_site(base_url, sitemap_url):
    # ── Step 1: Load and enforce robots.txt ───────────────────────
    rp = load_robots(base_url)

    # ── Step 2: Collect URLs from sitemap ─────────────────────────
    all_urls = get_urls_from_sitemap(sitemap_url)

    # ── Step 3: Filter — same domain, allowed, no binary assets ───
    filtered = [
        url for url in all_urls
        if urlparse(url).netloc == urlparse(base_url).netloc
        and not any(url.lower().endswith(ext) for ext in
                    [".pdf", ".jpg", ".jpeg", ".png", ".gif",
                     ".svg", ".zip", ".mp4", ".mp3", ".css", ".js"])
        and is_allowed(rp, url)   # ← robots.txt enforcement
    ]

    blocked = len(all_urls) - len(filtered)
    if blocked:
        print(f"🚫 {blocked} URLs removed by robots.txt or asset filter")

    print(f"\n🔗 {len(filtered)} pages to scrape (capped at {MAX_PAGES})\n")
    filtered = filtered[:MAX_PAGES]

    pages    = []
    pdf_seen = set()  # global dedup across all pages

    for i, url in enumerate(filtered):
        try:
            delay = random.uniform(MIN_DELAY, MAX_DELAY)
            print(f"⏳ Waiting {delay:.1f}s...")
            time.sleep(delay)

            response = requests.get(
                url,
                headers=HEADERS,
                timeout=REQUEST_TIMEOUT,
                stream=True
            )

            content_type = response.headers.get("Content-Type", "")
            if "text/html" not in content_type:
                print(f"⏭️  Skipping non-HTML: {url}")
                continue

            content_length = int(response.headers.get("Content-Length", 0))
            if content_length > MAX_PAGE_SIZE_KB * 1024:
                print(f"⏭️  Skipping oversized page: {url}")
                continue

            if response.status_code == 429:
                print("🛑 Rate limited — stopping early")
                break
            if response.status_code != 200:
                print(f"⚠️  HTTP {response.status_code}: {url}")
                continue

            soup = BeautifulSoup(response.text, "html.parser")
            for tag in soup(["script", "style", "nav", "footer",
                             "header", "aside", "form"]):
                tag.decompose()

            text = soup.get_text(separator=" ", strip=True)

            if len(text) > 200:
                page_entry = {"url": url, "text": text, "pdfs": []}

                # ── Step 4: Find and extract PDFs linked on this page ──
                pdf_links = find_pdf_links(soup, url, rp)
                for pdf_url in pdf_links:
                    if pdf_url in pdf_seen:
                        print(f"   ⏭️  PDF already extracted (dedup): {pdf_url}")
                        continue
                    pdf_seen.add(pdf_url)

                    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
                    pdf_text = extract_pdf_from_url(pdf_url)
                    if pdf_text:
                        page_entry["pdfs"].append({
                            "url":  pdf_url,
                            "text": pdf_text
                        })

                pages.append(page_entry)
                pdf_count = len(page_entry["pdfs"])
                pdf_note  = f" + {pdf_count} PDF(s)" if pdf_count else ""
                print(f"✅ ({i+1}/{len(filtered)}) Scraped: {url}{pdf_note}")

        except requests.exceptions.Timeout:
            print(f"⏱️  Timeout: {url}")
        except requests.exceptions.ConnectionError:
            print(f"🔌 Connection error: {url}")
        except Exception as e:
            print(f"⚠️  Error on {url}: {e}")

    return pages


# ── Run ─────────────────────────────────────────────────────────────
pages = scrape_site(BASE_URL, SITEMAP_URL)

with open("scraped.json", "w") as f:
    json.dump(pages, f, indent=2)

total_pdfs = sum(len(p["pdfs"]) for p in pages)
print(f"\n📄 Done. {len(pages)} pages + {total_pdfs} PDFs saved to scraped.json")

🤖 Fetching robots.txt: https://pvrea.coop/robots.txt
   robots.txt loaded successfully
📋 Fetching sitemap: https://pvrea.coop/sitemap_index.xml
📋 Fetching sitemap: https://pvrea.coop/post-sitemap.xml
   Found 154 URLs
📋 Fetching sitemap: https://pvrea.coop/page-sitemap.xml
   Found 93 URLs
   Found 247 URLs

🔗 247 pages to scrape (capped at 300)

⏳ Waiting 2.1s...
✅ (1/247) Scraped: https://pvrea.coop/the-co-op/news/big-data-bigger-demands-growth-done-pvrea-way/
⏳ Waiting 2.5s...
✅ (2/247) Scraped: https://pvrea.coop/the-co-op/news/planning-for-growth-power-you-can-count-on/
⏳ Waiting 2.5s...
✅ (3/247) Scraped: https://pvrea.coop/the-co-op/news/faces-places-voices-pvrea-george/
⏳ Waiting 2.2s...
✅ (4/247) Scraped: https://pvrea.coop/the-co-op/news/thank-you-ron-sutherland-for-32-years-of-service/
⏳ Waiting 2.7s...
✅ (5/247) Scraped: https://pvrea.coop/the-co-op/news/the-value-of-board-leadership-training-jan-peterson-2026/
⏳ Waiting 1.9s...
✅ (6/247) Scraped: https://pvrea.coop/the-co-

# Chunking Text

In [3]:
# chunk.py
import json

def chunk_text(text, chunk_size=150, overlap=20):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+chunk_size]))
        i += chunk_size - overlap
    return chunks

pages = json.load(open("scraped.json"))
all_chunks = []
html_chunks = 0
pdf_chunks  = 0

for page in pages:
    # ── HTML page chunks ──────────────────────────────────────────
    for chunk in chunk_text(page["text"]):
        all_chunks.append({
            "url":    page["url"],
            "source": "html",
            "text":   chunk
        })
        html_chunks += 1

    # ── PDF chunks (nested under their source page) ───────────────
    for pdf in page.get("pdfs", []):
        for chunk in chunk_text(pdf["text"]):
            all_chunks.append({
                "url":        pdf["url"],       # direct PDF URL for citation
                "source_page": page["url"],     # page the PDF was linked from
                "source":     "pdf",
                "text":       chunk
            })
            pdf_chunks += 1

with open("chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"✅ Total chunks : {len(all_chunks)}")
print(f"   HTML chunks  : {html_chunks}")
print(f"   PDF chunks   : {pdf_chunks}")
print(f"\n   Sample (HTML):\n{next(c for c in all_chunks if c['source'] == 'html')['text'][:300]}")
if pdf_chunks:
    print(f"\n   Sample (PDF):\n{next(c for c in all_chunks if c['source'] == 'pdf')['text'][:300]}")

✅ Total chunks : 3329
   HTML chunks  : 1412
   PDF chunks   : 1917

   Sample (HTML):
Big Data, Bigger Demands & Growth Done The PVREA Way Skip to Main Content Home ▸ The Co-op ▸ All News April 30, 2026 Big Data, Bigger Demands & Growth Done The PVREA Way PVREA News Big Data, Bigger Demands Many companies are choosing areas served by electric cooperatives for their data centers becau

   Sample (PDF):
Sales Tax Exempt Certificate Electricity & Gas for Industrial Use Note: If your company is 75% or more exempt from sales taxes, file this form with your utility company. The utility company will cease charging state and RTD/CD sales taxes and state collected local taxes. If your business is less tha


# Embedding

In [4]:
# embed.py
import json, chromadb
from sentence_transformers import SentenceTransformer

# Runs entirely on your machine — no server needed
model = SentenceTransformer("all-MiniLM-L6-v2")  # downloads once, ~80MB

chunks = json.load(open("chunks.json"))
texts  = [c["text"] for c in chunks]
urls   = [c["url"]  for c in chunks]

print(f"⏳ Embedding {len(texts)} chunks locally...")
embeddings = model.encode(texts, show_progress_bar=True).tolist()

db = chromadb.PersistentClient(path="./chroma_db")
collection = db.get_or_create_collection("company_docs")

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(texts))],
    metadatas=[{"url": u} for u in urls]
)

print(f"\n✅ Stored {collection.count()} chunks in ChromaDB")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Embedding 3329 chunks locally...


Batches:   0%|          | 0/105 [00:00<?, ?it/s]


✅ Stored 3329 chunks in ChromaDB


In [ ]:
# chatbot.py
import chromadb, requests, json
from sentence_transformers import SentenceTransformer

# Local embedding model (same one used in embed.py)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ChromaDB (local files)
db = chromadb.PersistentClient(path="./chroma_db")
collection = db.get_collection("company_docs")

LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"

def retrieve(query, top_k=5):
    embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=embedding, n_results=top_k)
    return results["documents"][0], results["metadatas"][0]

def ask(query):
    docs, metas = retrieve(query)
    context = "\n\n---\n\n".join(docs)
    context = context[:3000]  # ← add this — caps context at ~750 tokens
    sources = list(set(m["url"] for m in metas))

    payload = {
        "model": "local-model",
        "temperature": 0.3,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that answers questions about this company "
                    "using only the context below. If the answer isn't in the context, say "
                    "you don't know — do not make anything up.\n\n"
                    f"CONTEXT:\n{context}"
                )
            },
            {"role": "user", "content": query}
        ]
    }

    response = requests.post(LM_STUDIO_URL, json=payload)
    data = response.json()

    # ── Debug: show raw response if something goes wrong ──
    if "choices" not in data:
        print(f"\n⚠️  Unexpected response from LM Studio:")
        print(f"   Status code : {response.status_code}")
        print(f"   Response    : {data}")
        return

    answer = data["choices"][0]["message"]["content"]
    print(f"\n🤖 {answer}")
    print(f"\n📎 Sources: {', '.join(sources)}\n")

# --- Main loop ---
print("💬 Company Chatbot ready! Type 'quit' to exit.\n")
while True:
    q = input("You: ").strip()
    if q.lower() == "quit":
        break
    if q:
        ask(q)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


💬 Company Chatbot ready! Type 'quit' to exit.

